In [125]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import numpy as np
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
import json
import os

In [126]:
with open("../datas/couplet/vocabs", "r") as f:
    vocab = f.read().split('\n')
print(len(vocab))
with open('../datas/couplet/char2id.json', 'r', encoding='utf-8') as f:
    char2id = json.load(f)
with open('../datas/couplet/id2char.json', 'r', encoding='utf-8') as f:
    id2char = json.load(f)
print(len(char2id))
print(len(id2char))

9132
9132
9132


In [127]:
with open('../datas/couplet/train/out_id.json', 'r', encoding='utf-8') as f:
    train_out_id = json.load(f)
with open('../datas/couplet/train/in_id.json', 'r', encoding='utf-8') as f:
    train_in_id = json.load(f)
with open('../datas/couplet/test/out_id.json', 'r', encoding='utf-8') as f:
    test_out_id = json.load(f)
with open('../datas/couplet/test/in_id.json', 'r', encoding='utf-8') as f:
    test_in_id = json.load(f)
with open('../datas/couplet/train/true_len.json', 'r', encoding='utf-8') as f:
    train_true_len = json.load(f)
with open('../datas/couplet/test/true_len.json', 'r', encoding='utf-8') as f:
    test_true_len = json.load(f)
print(type(train_true_len))

<class 'list'>


In [128]:
class Encoder_RNN(nn.Module):
	def __init__(self,vocab_size,embed_dim,hidden_size,num_layers,bidirectional =False):
		super(Encoder_RNN, self).__init__()
		self.vocab_size = vocab_size
		self.hidden_size = hidden_size
		self.embed_dim = embed_dim
		self.num_layers = num_layers
		self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
		self.rnn = nn.RNN(embed_dim,hidden_size,num_layers,batch_first=True,bidirectional=bidirectional)
	def forward(self,x,true_len=None):
		if true_len is None:
			true_len = torch.full((x.size(0),), x.size(1),dtype=torch.long)
		embedded = self.embedding(x)
		packed = pack_padded_sequence(embedded,true_len,batch_first=True,enforce_sorted=False)
		output, hidden = self.rnn(packed)
		output,_ = pad_packed_sequence(output,batch_first=True,total_length=x.size(1),padding_value=0)
		hidden = hidden.permute(1,0,2).reshape(x.size(0),-1)
		return output, hidden

In [129]:
# # 编码器案例
# encoder = Encoder_RNN(100,128,256,3,bidirectional=False)
# token_ids = torch.randint(0, 20, size=(20,30))
# true_len = torch.randint(1, 10, size=(20,))
# output, hidden = encoder(token_ids,true_len)
# print(output.size())
# print(hidden.size())

In [130]:
class Decoder_RNN(nn.Module):
	def __init__(self,vocab_size,embed_dim,hidden_size,num_layers,encoder_bidirectional,encoder_hidden_size=None,encoder_num_layers=None):
		super(Decoder_RNN, self).__init__()
		encoder_directions = 2 if encoder_bidirectional else 1
		self.vocab_size = vocab_size
		self.hidden_size = hidden_size
		self.embed_dim = embed_dim
		self.num_layers = num_layers
		if encoder_hidden_size is None:
			encoder_hidden_size = hidden_size
		if encoder_num_layers is None:
			encoder_num_layers = num_layers
		self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
		self.rnn = nn.RNN(embed_dim,hidden_size,num_layers,batch_first=True,bidirectional=False)
		self.init_fc = nn.Linear(encoder_hidden_size * encoder_directions * encoder_num_layers, hidden_size * num_layers)
		self.fc = nn.Linear(hidden_size,vocab_size)

	def init_hidden(self,enc_hidden):
		hidden = self.init_fc(enc_hidden)
		hidden = hidden.reshape(hidden.size(0),self.num_layers,self.hidden_size).permute(1,0,2)
		return hidden.contiguous()

	def forward(self,x,hidden,true_len=None):
		if true_len is None:
			true_len = torch.full((x.size(0),), x.size(1),dtype=torch.long)
		embedded = self.embedding(x)
		packed = pack_padded_sequence(embedded,true_len,batch_first=True,enforce_sorted=False)
		output, hidden = self.rnn(packed,hidden)
		output,_ = pad_packed_sequence(output,batch_first=True,total_length=x.size(1),padding_value=0)
		score = self.fc(output)
		return score, hidden

	# def inference(self,x,hidden,max_len):
	# 	hidden = self.init_fc(hidden)
	# 	hidden = hidden.reshape(hidden.size(0),self.num_layers,self.hidden_size).permute(1,0,2)
	# 	for _ in range(max_len):
	# 		embedded = self.embedding(x)
	# 		out, hidden = self.rnn(embedded,hidden)
	# 		score = self.fc(out[:,-1,:])
	# 		pred = torch.argmax(score, dim=1, keepdim=True)
	# 		x = torch.cat([x, pred], dim=1)
	# 		if (pred == 2).all():
	# 			break
	# 	return x

In [131]:
# # 解码器案例
# decoder = Decoder_RNN(100, 128, 256,1,512,2,True)
# token_ids = torch.randint(0, 20, size=(20,10)) # 解码器的输入
# true_len = torch.randint(1, 10, size=(20,))
# h_n = torch.rand(20,512*2*2) # 编码器提取出来的特征向量
#
# decoder.train()
# decoder_score,hidden = decoder(token_ids, h_n,true_len)
# print(f"解码器输出:{decoder_score.shape}")
# print(f"解码器输出:{hidden.shape}")
# print(decoder_score.argmax(2).shape)
# print(decoder_score.argmax(2))

In [132]:
class Seq2Seq_RNN(nn.Module):
	def __init__(self,encoder,decoder):
		super(Seq2Seq_RNN, self).__init__()
		self.encoder = encoder
		self.decoder = decoder

	def forward(self,x,y,x_true_len=None,y_true_len=None):
		output, enc_hidden = self.encoder(x,x_true_len)
		dec_hidden = self.decoder.init_hidden(enc_hidden)
		score, _= self.decoder(y,dec_hidden,y_true_len)
		return score

	@torch.no_grad()
	def generate(self,x,sos_id,eos_id,x_true_len=None,max_len=50):
		bs = x.size(0)
		output, enc_hidden = self.encoder(x, x_true_len)
		dec_hidden = self.decoder.init_hidden(enc_hidden)
		# 起始 token
		inp = torch.full((bs,1),sos_id,dtype=torch.long,device=device)
		results = []
		finished = torch.zeros(bs,dtype=torch.bool,device=device)
		for _ in range(max_len):
			true_len = torch.ones(bs, dtype=torch.long)
			output,dec_hidden = self.decoder(inp,dec_hidden,true_len)
			pred = output.argmax(dim=-1)
			results.append(pred)
			finished |= (pred.squeeze(1) == eos_id)
			if finished.all():
				break
			inp = pred
		return torch.cat(results,dim=1)

In [133]:
# # seq2seq案例
# seq2seq = Seq2Seq_RNN(100,128,256,3,True,256,512,2)
#
# train_ids = torch.randint(0, 20, size=(20,20))
# test_ids = torch.randint(0, 20, size=(20,20))
#
# score = seq2seq(token_ids,test_ids)
# print(f"输出:{score.shape}")

In [134]:
class MyDataset(Dataset):
    def __init__(self, data, target, true_lens):

        data = np.array(data).astype('int64')
        target = np.array(target).astype('int64')
        true_lens = np.array(true_lens).astype('int64')

        self.data = torch.from_numpy(data)
        self.target = torch.from_numpy(target)
        self.true_lens = torch.from_numpy(true_lens)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.target[idx],self.true_lens[idx]

In [135]:
print(len(train_in_id))
print(len(test_in_id))
train_in_id = train_in_id[:20000]
train_out_id = train_out_id[:20000]
test_in_id = test_in_id
test_out_id = test_out_id

770492
4001


In [136]:
vocab_size = len(vocab)
embed_dim = 128
hidden_size = 256
num_layers = 2
bidirectional = False

batch_size = 64
learning_rate = 0.01
epochs = 5
device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(device)

mps


In [137]:
X_train,y_train,X_test,y_test = train_in_id,train_out_id,test_in_id,test_out_id
ture_lens_train,ture_lens_test = train_true_len,test_true_len
train_loader = DataLoader(MyDataset(X_train,y_train,ture_lens_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MyDataset(X_test,y_test,ture_lens_test), batch_size=batch_size//2, shuffle=False)

In [138]:
encoder = Encoder_RNN(vocab_size,embed_dim,hidden_size,num_layers,bidirectional).to(device)
decoder = Decoder_RNN(vocab_size,embed_dim,hidden_size,num_layers,bidirectional).to(device)
model = Seq2Seq_RNN(encoder,decoder).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=0).to(device)

In [139]:
summary_dir = "../datas/couplet/summary"
best_model_path = "../datas/couplet/model/best_model.pkl"
last_model_path = "../datas/couplet/model/last_model.pkl"

writer = SummaryWriter(log_dir=summary_dir)
start_epoch = 0
best_acc = 0.0
resume_training = True

if os.path.exists(last_model_path) and resume_training:
	print(f"发现历史保存模型 '{last_model_path}'，正在加载...")
	checkpoint = torch.load(last_model_path,map_location=device,weights_only=False)
	model.load_state_dict(checkpoint['model'])
	optimizer.load_state_dict(checkpoint['optimizer'])
	start_epoch = checkpoint["epoch"] + 1
	best_acc = checkpoint['best_acc']
	print(f"成功恢复训练，将从 Epoch {start_epoch} 开始，历史最佳准确率: {best_acc:.2f}%\n")

for epoch in range(start_epoch,start_epoch+epochs):
	model.train()
	train_loss_sum = 0.0

	train_pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}/{start_epoch+epochs-1}')
	for batch_idx, (data, target, true_len) in enumerate(train_pbar):
		data,target,true_len = data.to(device), target.to(device),true_len.cpu()
		optimizer.zero_grad()
		output = model(data,target,true_len,true_len)
		loss = criterion(output.permute(0,2,1), target)
		loss.backward()
		optimizer.step()

		train_loss_sum += loss.item()
		train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
		global_step = epoch * len(train_loader) + batch_idx
		writer.add_scalar('Train/Step_Loss', loss.item(), global_step)

	avg_train_loss = train_loss_sum / len(train_loader)
	writer.add_scalar('Train/Avg_Loss', avg_train_loss, epoch)

	model.eval()
	test_loss = 0.0
	correct = 0
	total_tokens = 0

	test_pbar = tqdm(test_loader, desc=f'Test Epoch  {epoch}/{start_epoch+epochs-1}', leave=False)
	with torch.no_grad():
		for data, target, true_len in test_pbar:
			data, target, true_len = data.to(device), target.to(device), true_len.cpu()
			output = model(data,target,true_len, true_len)

			# 累加 batch_loss
			batch_loss = criterion(output.permute(0,2,1), target).item()
			test_loss += batch_loss

			pred = output.argmax(dim=-1)
			mask = (target != 0)
			correct += pred.eq(target).masked_select(mask).sum().item()
			total_tokens += mask.sum().item()

			test_pbar.set_postfix({'loss': f'{batch_loss:.4f}'})

	# 计算平均测试损失和准确率（修正了原代码中的逻辑）
	avg_test_loss = test_loss / len(test_loader)
	accuracy = 100. * correct / total_tokens

	# 将测试指标记录到 TensorBoard
	writer.add_scalar('Test/Epoch_Loss', avg_test_loss, epoch)
	writer.add_scalar('Test/Accuracy', accuracy, epoch)

	print(f'Epoch {epoch} 完成 | Test Avg Loss: {avg_test_loss:.4f} | Accuracy: {correct}/{total_tokens} ({accuracy:.2f}%)')

	checkpoint = {
		'epoch': epoch,
		'model': model.state_dict(),
		'optimizer': optimizer.state_dict(),
		'best_acc': best_acc
	}
	if epoch == start_epoch+epochs-1:
		print("保存本轮最后一个模型")
		torch.save(checkpoint, last_model_path)

	if accuracy > best_acc:
		print(f'>>> 发现新的最佳准确率: {accuracy:.2f}% (前最佳: {best_acc:.2f}%)，正在保存最佳模型...\n')
		best_acc = accuracy
		checkpoint['best_acc'] = best_acc  # 更新 checkpoint 里的最佳记录
		torch.save(checkpoint, best_model_path)
	else:
		print() # 输出空行以便于阅读

writer.close()
print("训练全部完成！")



Train Epoch 0/4:   0%|          | 0/313 [00:00<?, ?it/s]

Test Epoch  0/4:   0%|          | 0/126 [00:00<?, ?it/s]

Epoch 0 完成 | Test Avg Loss: 0.0557 | Accuracy: 45311/45546 (99.48%)
>>> 发现新的最佳准确率: 99.48% (前最佳: 0.00%)，正在保存最佳模型...



Train Epoch 1/4:   0%|          | 0/313 [00:00<?, ?it/s]

Test Epoch  1/4:   0%|          | 0/126 [00:00<?, ?it/s]

...

Test Epoch  3/4:   0%|          | 0/126 [00:00<?, ?it/s]

Epoch 3 完成 | Test Avg Loss: 0.0494 | Accuracy: 45392/45546 (99.66%)
>>> 发现新的最佳准确率: 99.66% (前最佳: 99.65%)，正在保存最佳模型...



Train Epoch 4/4:   0%|          | 0/313 [00:00<?, ?it/s]

Test Epoch  4/4:   0%|          | 0/126 [00:00<?, ?it/s]

Epoch 4 完成 | Test Avg Loss: 0.0496 | Accuracy: 45392/45546 (99.66%)
保存本轮最后一个模型

训练全部完成！


In [141]:
def text2id(text,max_len):
	text_id = []
	true_len = []
	for sentence in text:
		true_len.append(len(sentence)+2)
		temp = [char2id[char] if char in char2id else char2id['<unk>'] for char in sentence]
		temp = [char2id["<s>"]] + temp + [char2id['</s>']]

		if len(temp) < max_len:
			temp = temp + [0]*(max_len-len(temp))
		elif len(temp) > max_len:
			temp = temp[:max_len]
		text_id.append(temp)
	return text_id, true_len


In [142]:
def ids_to_text(ids):
	"""将 token id 列表转回文字，跳过特殊 token"""
	tokens = []
	for i in ids:
		w = id2char.get(str(i),'')
		if w in ('', '<p>', '</s>'):
			break
		if w == '<s>':
			continue
		tokens.append(w)
	return ''.join(tokens)

def predict(model,src_sentences,sos_id,eos_is,true_len,device,max_len=50):
	"""
    src_sentences: List[str]，每个字符串是一句上联（字级别，空格分隔或直接字符串均可）
    返回: List[str] 下联列表
    """
	model.eval()
	src_sentences = torch.tensor(src_sentences,dtype=torch.long).to(device)
	true_len = torch.tensor(true_len,dtype=torch.long)
	with torch.no_grad():
		result_ids = model.generate(src_sentences,sos_id,eos_is,true_len,max_len)
	result_ids = result_ids.cpu().tolist()
	return [ids_to_text(ids) for ids in result_ids]

In [143]:
sos_id = char2id['<s>']
eos_id = char2id['</s>']

# 加载最佳模型进行推理
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model'])

test_inputs = [
    "晚风摇树树还挺",
    "春风送暖入屠苏",
    "两个黄鹂鸣翠柳",
]

test_inputs_splited = [[j for j in i] for i in test_inputs]
max_len = max(len(sentence) for sentence in test_inputs_splited)
max_len +=2

test_in_id,test_true_len = text2id(test_inputs,max_len)
print(test_in_id,test_true_len)

outputs = predict(model, test_in_id, sos_id, eos_id,test_true_len,device,10)

print("\n===== 推理结果 =====")
for src, tgt in zip(test_inputs, outputs):
    print(f"上联：{src}")
    print(f"下联：{tgt}")
    print()


[[1, 487, 6, 509, 153, 153, 374, 1581, 2], [1, 7, 6, 372, 328, 118, 2447, 966, 2], [1, 158, 796, 167, 2670, 419, 235, 67, 2]] [9, 9, 9]

===== 推理结果 =====
上联：晚风摇树树还挺
下联：饫饫饫饫饫饫饫饫饫

上联：春风送暖入屠苏
下联：饫饫饫饫饫饫饫饫饫

上联：两个黄鹂鸣翠柳
下联：饫饫饫饫饫饫饫饫饫

